## Step 1 — Load votes, jurors, show order, and pot assignments

Inputs:
- `jury_votes.xlsx`: long-format votes with `Voting_country`, `Juror`, `Participating_country`, `Rank`, `DoB`, `ShowOrder`.
- `SF1_-_voting_countries.csv`: pot assignment per voting country.

Helpers live in `jury_helpers.py`.

In [1]:
import pandas as pd
from jury_helpers import (
    DICT_ISO, ISO_TO_COUNTRY,
    rank_to_exp_score, rank_to_points,
    rank_jury_with_tiebreakers,
    rank_final_classification,
    apply_pot_substitutes,
)

In [2]:
INPUT_FILE = r'jury_votes.xlsx'
POTS_FILE  = r'SF1_-_voting_countries.csv'

long_df = pd.read_excel(INPUT_FILE)
for col in ['Voting_country', 'Participating_country']:
    unknown = set(long_df[col]) - set(ISO_TO_COUNTRY)
    if unknown:
        raise ValueError(f'Unknown ISO codes in {col}: {sorted(unknown)}')
    long_df[col] = long_df[col].map(ISO_TO_COUNTRY)

long_df.head()

,Voting_country,Juror,Participating_country,Rank,DoB,ShowOrder
0,Croatia,Juror 1,Croatia,0,1995-07-02,12
1,Croatia,Juror 2,Croatia,0,1969-07-20,12
2,Croatia,Juror 3,Croatia,0,1989-06-09,12
3,Croatia,Juror 4,Croatia,0,1997-09-10,12
4,Croatia,Juror 5,Croatia,0,1997-05-23,12


In [3]:
pots_df = pd.read_csv(POTS_FILE, sep=';', skiprows=1)
pots_df = pots_df[['Country', 'ISO', 'SF1_Pot']].dropna()
pots_df['SF1_Pot'] = pots_df['SF1_Pot'].astype(int)
pot_assignments = dict(zip(pots_df['Country'], pots_df['SF1_Pot']))
pot_assignments

{'Croatia': 1,
 'Finland': 1,
 'Montenegro': 1,
 'Serbia': 1,
 'Sweden': 1,
 'Belgium': 2,
 'Georgia': 2,
 'Israel': 2,
 'Moldova': 2,
 'Poland': 2,
 'Estonia': 3,
 'Greece': 3,
 'Lithuania': 3,
 'Portugal': 3,
 'San Marino': 3,
 'Germany': 0,
 'Italy': 0}

In [4]:
participating_order = long_df['Participating_country'].drop_duplicates().tolist()
voting_order        = long_df['Voting_country'].drop_duplicates().tolist()
juror_order         = long_df['Juror'].drop_duplicates().tolist()

ranks_df = (
    long_df
    .pivot(index='Participating_country',
           columns=['Voting_country', 'Juror'],
           values='Rank')
    .reindex(index=participating_order,
             columns=pd.MultiIndex.from_product(
                 [voting_order, juror_order],
                 names=['Voting_country', 'Juror']))
)
ranks_df.head()

Voting_country        Croatia                                                  \
Juror                 Juror 1 Juror 2 Juror 3 Juror 4 Juror 5 Juror 6 Juror 7   
Participating_country                                                           
Croatia                     0       0       0       0       0       0       0   
Finland                    11       6       9       1      13       7      14   
Montenegro                 12       2       3      11       2       5       5   
Serbia                      2       4       8      10      12      13      10   
Sweden                      9       7       4      13       4      14      13   

Voting_country        Finland                  ... Germany                  \
Juror                 Juror 1 Juror 2 Juror 3  ... Juror 5 Juror 6 Juror 7   
Participating_country                          ...                           
Croatia                    12       7       5  ...      13       3       1   
Finland                     0       0       0  ...       1      11      15   
Montenegro                  6       4      11  ...       3       8       2   
Serbia                     11       5       2  ...       9      10       4   
Sweden                      7       6       7  ...      10       9       8   

Voting_country          Italy                                                  
Juror                 Juror 1 Juror 2 Juror 3 Juror 4 Juror 5 Juror 6 Juror 7  
Participating_country                                                          
Croatia                    10      12      11       9       3       7      11  
Finland                     2       7       6      12       2       1      14  
Montenegro                 14      15      14       1      14      12       9  
Serbia                      8       4       5       6       4       4      15  
Sweden                      7      10      10      15      11       9       3  

[5 rows x 119 columns]

In [5]:
juror_dobs = (
    long_df.drop_duplicates(['Voting_country', 'Juror'])
           .set_index(['Voting_country', 'Juror'])['DoB']
)
show_order = (
    long_df.drop_duplicates('Participating_country')
           .set_index('Participating_country')['ShowOrder']
           .to_dict()
)
show_order

{'Croatia': 12,
 'Finland': 9,
 'Montenegro': 1,
 'Serbia': 13,
 'Sweden': 14,
 'Belgium': 11,
 'Georgia': 6,
 'Israel': 10,
 'Moldova': 2,
 'Poland': 8,
 'Estonia': 3,
 'Greece': 4,
 'Lithuania': 15,
 'Portugal': 5,
 'San Marino': 7}

## Step 2 — Convert ranks to exponential scores

In [6]:
exp_scores_df = ranks_df.map(rank_to_exp_score)
exp_scores_df.head()

Voting_country         Croatia                                                \
Juror                  Juror 1  Juror 2  Juror 3   Juror 4  Juror 5  Juror 6   
Participating_country                                                          
Croatia                0.00000  0.00000  0.00000   0.00000  0.00000  0.00000   
Finland                1.79482  4.64089  2.62454  12.00000  1.22741  3.83783   
Montenegro             1.48425  9.92351  8.20634   1.79482  9.92351  5.61200   
Serbia                 9.92351  6.78631  3.17373   2.17039  1.48425  1.22741   
Sweden                 2.62454  3.83783  6.78631   1.22741  6.78631  1.01502   

Voting_country                  Finland                    ...   Germany  \
Juror                  Juror 7  Juror 1  Juror 2  Juror 3  ...   Juror 5   
Participating_country                                      ...             
Croatia                0.00000  1.48425  3.83783  5.61200  ...   1.22741   
Finland                1.01502  0.00000  0.00000  0.00000  ...  12.00000   
Montenegro             5.61200  4.64089  6.78631  1.79482  ...   8.20634   
Serbia                 2.17039  1.79482  5.61200  9.92351  ...   2.62454   
Sweden                 1.22741  3.83783  4.64089  3.83783  ...   2.17039   

Voting_country                              Italy                              \
Juror                  Juror 6   Juror 7  Juror 1  Juror 2  Juror 3   Juror 4   
Participating_country                                                           
Croatia                8.20634  12.00000  2.17039  1.48425  1.79482   2.62454   
Finland                1.79482   0.83938  9.92351  3.83783  4.64089   1.48425   
Montenegro             3.17373   9.92351  1.01502  0.83938  1.01502  12.00000   
Serbia                 2.17039   6.78631  3.17373  6.78631  5.61200   4.64089   
Sweden                 2.62454   3.17373  3.83783  2.17039  2.17039   0.83938   

Voting_country                                     
Juror                  Juror 5   Juror 6  Juror 7  
Participating_country                              
Croatia                8.20634   3.83783  1.79482  
Finland                9.92351  12.00000  1.01502  
Montenegro             1.01502   1.48425  2.62454  
Serbia                 6.78631   6.78631  0.83938  
Sweden                 1.79482   2.62454  8.20634  

[5 rows x 119 columns]

## Step 3 — Sum per jury, rank with tie-breakers, award points

Within-jury tie-breaker chain (when two countries share the same exponential sum):
1. Majority of better individual rankings among that jury's jurors.
2. Vote of the youngest juror.
3. Show of hands — interactive prompt for the winning ISO code.

In [7]:
jury_sums = exp_scores_df.T.groupby(level='Voting_country', sort=False).sum().T

jury_sums_for_ranking = jury_sums.copy()
for vc in jury_sums_for_ranking.columns:
    if vc in jury_sums_for_ranking.index:
        jury_sums_for_ranking.loc[vc, vc] = pd.NA

In [8]:
jury_ranks = pd.DataFrame(
    index=jury_sums.index, columns=jury_sums.columns, dtype='Int64'
)

for vc in jury_sums.columns:
    sums = jury_sums_for_ranking[vc]
    ranks_in_jury = ranks_df.xs(vc, axis=1, level='Voting_country')
    dobs_for_jury = juror_dobs.loc[vc].to_dict()
    rank_series = rank_jury_with_tiebreakers(
        vc, sums, ranks_in_jury, dobs_for_jury
    )
    for c, r in rank_series.items():
        jury_ranks.loc[c, vc] = r

jury_points = jury_ranks.map(rank_to_points).astype(int)
jury_points

Voting_country,Croatia,Finland,Montenegro,Serbia,Sweden,Belgium,Georgia,Israel,Moldova,Poland,Estonia,Greece,Lithuania,Portugal,San Marino,Germany,Italy
Participating_country,,,,,,,,,,,,,,,,,
Croatia,0,1,4,5,12,0,0,8,4,6,7,0,12,0,1,10,0
Finland,2,0,0,7,5,8,0,0,0,0,8,5,8,1,3,3,10
Montenegro,10,6,0,12,0,12,5,5,0,7,6,10,0,0,0,6,0
Serbia,1,0,1,0,3,4,6,12,1,5,4,4,0,0,2,0,6
Sweden,0,0,0,8,0,6,1,1,6,1,10,3,0,8,4,0,0
Belgium,12,12,3,2,1,0,0,7,10,0,3,7,4,10,6,0,7
Georgia,6,4,8,0,6,0,0,0,12,0,0,1,0,4,0,5,12
Israel,3,0,12,0,7,7,8,0,0,4,0,0,5,3,7,1,0
Moldova,8,5,0,0,0,0,7,2,0,0,5,2,10,7,12,7,0


## Step 3.5 — Pot substitution for invalid juries

A national jury is invalid if it has fewer than 3 valid jurors (per EBU §1.3) or if it has been disqualified by the EBU. Add country names to `manual_disqualified` to force pot substitution for testing.

When invalid juries are present, this step (per EBU §4.3):
1. Identifies the disqualified country's pot.
2. Averages the exponential rank values from each individual juror in the remaining pot members (zeros / home votes excluded).
3. Falls back to Pot 0 if only ≤1 valid pot member remains.
4. Tie-breaks (rank-1 count → rank-2 → … → show order).
5. Awards 12-10-8-…-1 to the top 10 non-disqualified countries.

If no juries are invalid, the cell prints a confirmation and `jury_points` passes through unchanged.

In [9]:
manual_disqualified = []   # add country names here to force pot substitution

jury_points, substituted = apply_pot_substitutes(
    jury_points, ranks_df, pot_assignments, show_order,
    long_df, manual_disqualified=manual_disqualified,
)

Checking national juries for validity...
All national juries valid (≥ 3 jurors each, none disqualified).

No pots used — final classification proceeds unchanged.


## Step 4 — Final classification with tie-breakers

Final-tie chain (when two countries have the same total points):
1. Highest number of juries that gave any points.
2. Highest number of 12-point scores.
3. Walk down — most 10s, 8s, 7s, 6s, 5s, 4s, 3s, 2s, 1s.
4. Earlier in the show running order wins.

In [10]:
final_classification = rank_final_classification(jury_points, show_order)
final_classification

  [Final] Tie at total=78 points: Portugal vs Lithuania
      juries giving points: Portugal=12, Lithuania=13
    resolved: Lithuania ranks above Portugal


,Total_points,Rank
Participating_country,,
Belgium,84,1
Montenegro,79,2
Lithuania,78,3
Portugal,78,4
Poland,73,5
Croatia,70,6
Estonia,69,7
Moldova,65,8
Greece,64,9
